In [1]:
from pathlib import Path
import sys, json, hashlib
ROOT = Path.cwd().resolve().parent
sys.path.insert(0, str(ROOT / 'src'))
from sneak.fcg_registry import FCGRegistry, verify_rows
print('Root:',ROOT)

Root: /Users/byron/projects/hackathon/sneak-audit-fco-20261020


In [2]:
r = FCGRegistry(ROOT / 'data/fcg/sneak-016-replay.jsonl')
rows = r.rows()
assert len(rows) == 58
p55=verify_rows(rows[:55]); p57=verify_rows(rows[:57]); current=r.verify()
assert p55['mmr_root']=='089e49c083f1d6170402a366cfa76f3b939cbdcefce262ff39a732061463dadc'
assert p57['mmr_root']=='9b2bc310380b6d9fd586caf0cbe6b185d7d4c95b41c95c31e31231123861f416'
assert current['mmr_root']=='f0579f511bccecf09c588f1d564e33a12af3409502a56b7868c4b61eab87587a'
print('PASS prefixes and final',current)

PASS prefixes and final {'verified': True, 'fco_count': 58, 'mmr_size': 112, 'mmr_root': 'f0579f511bccecf09c588f1d564e33a12af3409502a56b7868c4b61eab87587a', 'fcg_edges': 67}


In [3]:
live=json.loads((ROOT/'receipts/connectors/SNEAK-018_R15_LIVE_REVIEW.json').read_text())
read=json.loads((ROOT/'receipts/connectors/SNEAK-019_BAND_READBACK.json').read_text())
assert live['cognee']['exact_digest_recall'] is True
assert live['band']['post_ok'] is True
assert read['payload']['message_id_present'] is True
assert read['payload']['payload_digest_present'] is True
assert live['payload_sha256']==read['payload']['canonical_packet_sha256']
assert read['successor_mmr']['mmr_root']==current['mmr_root']
assert r.inclusion_proof(rows[-1]['fco_id'])['verified']
print('PASS bound readback, inclusion; agent acknowledgment',live['band']['agent_acknowledgment'])

PASS bound readback, inclusion; agent acknowledgment NOT_OBSERVED


In [4]:
scans=[row['identity'] for row in rows if row['identity']['kind']=='scan']
scores=[row['identity'] for row in rows if row['identity']['kind']=='score']
assert len(scans)==8 and all(row['state']=='NOT_TESTED' for row in scans)
assert len(scores)==8 and all(row['state']=='NULL' for row in scores)
assert current['fco_count']==58 and current['fcg_edges']==67
print('PASS: old failure/null states unchanged, 58 FCOs and 67 edges')

PASS: old failure/null states unchanged, 58 FCOs and 67 edges
